# RAG with Azure AI Search

<img src="./architecture.png">

<img src="vector_embeddings.svg" >


### Installing Packages

In [1]:
%pip install openai



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Loading variables from the .env file

### ENVIRONMENT VARIABLES 

AZURE_SEARCH_SERVICE_ENDPOINT=https://jaganjamiaisearch.search.windows.net

AZURE_SEARCH_INDEX_NAME=jaganjami

AZURE_SEARCH_API_KEY=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

EMBEDDING_ENGINE=text-embedding-ada-002

AZURE_OPENAI_ENDPOINT=https://jaganjami01.openai.azure.com/

AZURE_OPENAI_KEY=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

GPT_ENGINE=gpt-4o

In [2]:
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
key = os.getenv("AZURE_SEARCH_API_KEY")

### Creating an Azure OpenAI Client

In [3]:
from openai import AzureOpenAI

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")  
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating Embedding Generator Function

In [4]:
def generate_embeddings(client, text):
    embedding_model = os.getenv("EMBEDDING_ENGINE")
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

In [5]:
user_query = "What is the review of the creek hotel in Dubai?"
vectorised_user_query = generate_embeddings(azure_openai_client, user_query)
print(vectorised_user_query)


[0.01658058539032936, -0.004367268644273281, 0.014174828305840492, -0.012582381255924702, -0.024850374087691307, 0.015856124460697174, -0.018521593883633614, -0.01555540505796671, 0.001976889558136463, -0.017769794911146164, -0.005754679441452026, -0.004346765112131834, -0.0029303075280040503, -0.015746772289276123, -0.0002310927666258067, 0.0007265966269187629, 0.02371584065258503, -0.020694976672530174, 0.012445691041648388, -0.012425187043845654, -0.03906621038913727, 0.01875396817922592, -0.006503061391413212, -0.031493544578552246, -0.0012498658616095781, 0.012486698105931282, 0.0017889399314299226, -0.017605766654014587, -0.0099442508071661, -0.00043164368253201246, 0.0076615153811872005, -0.003049911931157112, 0.0011610168730840087, 0.010019429959356785, -0.0012763496488332748, 0.020325912162661552, -0.008235616609454155, 0.008563674055039883, 0.02517843246459961, 0.008508997969329357, 0.008584178052842617, -0.014789937064051628, -0.011502524837851524, -0.006974644493311644, -0.

In [6]:
context=[]

### Sending API call to the Search Index

In [7]:
import requests
import json


url = f"{service_endpoint}/indexes/{index_name}/docs/search?api-version=2023-11-01"
    
headers = {
        "Content-Type": "application/json",
        "api-key": key
    }
    
body =   {
        "count": True,
        "select": "chunk",
        "vectorQueries": [
            {
                "vector": vectorised_user_query,
                "k": 3,
                "fields": "text_vector",
                "kind": "vector"
            }
        ]
    }
    
response = requests.post(url, headers=headers, data=json.dumps(body))
documents = response.json()['value']

for doc in documents:
    context.append(dict(
        {
            "chunk": doc['chunk'],
            "score": doc['@search.score']
            
        }
    ))
    
for doc in context:
    print(doc)



{'chunk': "Average place - great location \n\nThe Creek Hotel, Dubai, UAE \n\n1/3/2018 \n\nI stayed here twice last year; room was booked by a company I was doing business with in DXB.First \nof all, the hotel is in a great location in the middle of Bur Dubai. Lots of shopping nearby, many \nrestaurants, relatively close to the Creek, easy to get taxis. Within walking distance to a lot. Not sure \nof how easy it is to get to now (due to construction, etc). But not bad in last year.The hotel is one of \nthe older places in Dubai, and looks it. Kind of like a 1970s Holiday Inn that has not been updated. \nReception was efficient, if a bit indifferent. The porter was good. I had a one BR once and a one BR \nsuite once. Both were clean, if a bit worn. About what you'd expect from an older 3*. No problems \nwith housekeeping; clean sheets and linens. The one BR had a bed, OK mattress, dresser, desk and \nnightstand. The suite had a BR (just about the same as the one BR) and a good sized liv

### Calling GPT Engine for Summarisation

In [9]:
system_prompt = f""""You are meant to behave as a RAG chatbot that derives its context from a database of hotel reviews stored in Azure AI Search Solution.
please answer strictly from the context from the database provided and if you dont have an answer please politely say so. dont include any extra 
information that is not in the context and dont include links as well.
the context passed to you will be in the form of a pythonic list with each object in the list containing details of hotel reviews and
having structure as follows:

 "chunk": "the content of the review",
 "score": "the relevancy score of the review"


the pythonic list contains best 3 matches to the user query based on cosine similarity of the embeddings of the user query and the review descriptions.
please structure your answers in a very professional manner and in such a way that the user does not get to know that its RAG working under the hood
and its as if they are talking to a human. """

user_prompt = f""" the user query is: {user_query}
the context is : {context}"""

chat_completions_response = azure_openai_client.chat.completions.create(
    model = os.getenv("GPT_ENGINE"),
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)

print(chat_completions_response.choices[0].message.content)



The Creek Hotel in Dubai has received mixed reviews from guests. Here is an overview based on the feedback:

1. **Location and Accessibility**: The hotel is situated in a great location in Bur Dubai, close to shopping areas, restaurants, and the Creek. Guests find it convenient for getting taxis and within walking distance to many attractions. However, there might be some challenges due to construction in the area.

2. **Rooms and Furnishings**: The hotel is one of the older establishments in Dubai, with furnishings that are slightly dated. Rooms are clean but worn, and the standard is comparable to a 2-3 star property. The size of the rooms and suites is appreciated for the price, offering decent space and basic amenities.

3. **Services and Facilities**: Reception service is efficient, though sometimes indifferent. Housekeeping is reliable, and the porter service is commendable. The restaurant offers a limited but satisfying menu, and the pub is noted for its quality. However, intern